# Stage 1 — Prepare
**Twitter Sentiment Analysis MLOps · MAI201**

Notebook version of `src/prepare.py` (DVC stage `prepare`).

Loads the raw Sentiment140 CSV, cleans the tweet text, and writes a stratified
train/test split to `data/processed/`.

**Input:** `data/raw/sentiment140.csv` · **Outputs:** `data/processed/train.csv`, `data/processed/test.csv`

> Run this notebook from the **project root** (same folder as `params.yaml`).

In [ ]:
import os, re
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split

# Load config — same params.yaml that drives the DVC pipeline
with open("params.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)["prepare"]
cfg

In [ ]:
RAW_COLUMNS = ["target", "id", "date", "flag", "user", "text"]

raw_path = cfg["raw_path"]
assert os.path.exists(raw_path), (
    f"Dataset not found at {raw_path}. Download Sentiment140 from Kaggle "
    "and place the CSV there (see README Step 5).")

df = pd.read_csv(raw_path, encoding="latin-1", header=None, names=RAW_COLUMNS)
print(f"Loaded {len(df):,} rows")
df.head(3)

## Labels
Sentiment140: `0` = negative, `4` = positive (no neutral in the released set) → binary `label` column.

In [ ]:
df = df[df["target"].isin([0, 4])].copy()
df["label"] = (df["target"] == 4).astype(int)
df["label"].value_counts()

## Optional subsampling
`sample_size` in `params.yaml` keeps iteration fast (stratified). Set it to `null` for the full 1.6M final run.

In [ ]:
sample_size = cfg.get("sample_size")
if sample_size:
    df = (df.groupby("label", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), sample_size // 2),
                                      random_state=cfg["random_state"]))
            .reset_index(drop=True))
print(f"Working set: {len(df):,} rows")

## Text cleaning
Lowercase, strip URLs and @mentions, drop the `#` symbol, normalize whitespace.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
WS_RE = re.compile(r"\s+")

def clean_text(t: str) -> str:
    if cfg.get("lowercase", True):      t = t.lower()
    if cfg.get("remove_urls", True):    t = URL_RE.sub(" ", t)
    if cfg.get("remove_mentions", True):t = MENTION_RE.sub(" ", t)
    if cfg.get("remove_hashtag_symbol", True): t = t.replace("#", " ")
    return WS_RE.sub(" ", t).strip()

df["text"] = df["text"].astype(str).map(clean_text)
df = df[df["text"].str.len() > 0][["text", "label"]]
df.head(3)

## Stratified train/test split + save

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=cfg["test_size"],
    random_state=cfg["random_state"], stratify=df["label"])

os.makedirs("data/processed", exist_ok=True)
train_df.to_csv("data/processed/train.csv", index=False)
test_df.to_csv("data/processed/test.csv", index=False)

print(f"train.csv: {len(train_df):,} rows   test.csv: {len(test_df):,} rows")
print(f"positive share (train): {train_df['label'].mean():.3f}")